# 4. Devemos Trocar de Modelo?

## Comparativo do desempenho global

Como temos dados categóricos pareados, o teste que pareceu melhor se encaixar foi de McNemar.

Para o teste, vamos assumir que a H0 seja: os modelos produzem resultados iguais.

In [50]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
import pandas as pd

In [51]:
dados = pd.read_csv('../dados/chamados_com_predicoes.csv', index_col='id_chamado')

In [52]:
acerto_a = (dados["categoria_real"] == dados["pred_modelo_a"]).astype(int)
acerto_b = (dados["categoria_real"] == dados["pred_modelo_b"]).astype(int)

In [53]:
b = ((acerto_a == 1) & (acerto_b == 0)).sum()
c = ((acerto_a == 0) & (acerto_b == 1)).sum() 

In [54]:
b

np.int64(517)

In [55]:
c

np.int64(991)

In [56]:
ambos_certos = ((acerto_a == 1) & (acerto_b == 1)).sum()
ambos_errados = ((acerto_a == 0) & (acerto_b == 0)).sum()

In [57]:
tabela = [[ambos_certos, b],
          [c, ambos_errados]]

In [58]:
resultado = mcnemar(tabela, exact=False) #False é recomendado para grandes amostras, True para pequenas amostras
print(f"Estatística: {resultado.statistic}")
print(f"p-valor: {resultado.pvalue}")

Estatística: 148.36140583554376
p-valor: 3.954851106570505e-34


$$\chi^2 = \frac{(|b - c| - 1)^2}{b + c}$$

O p-value indica que a probabilidade desse caso ou de algum mais extremo acontecer dado que H0 é verdadeira é 3.954851106570505e-34, ou seja, extremamente baixa. Portanto podemos globalmente negar H0

H1 vai nos dizer que a performance dos modelos é diferente, mas nesse caso gostaríamos de saber efetivamente qual modelo é melhor. Olhando a fórmula acima, percebemos que é preciso analisar os valores de b e c para conseguir responder essa pergunta. Como C é 991 e b é 517, então globalmente o modelo b se mostrou melhor.

# Análise de desempenho por categoria.

Desejamos agora aplicar o mesmo teste por categoria para comparar o desempenho dos dois modelos em cada categoria.

In [59]:
def mcnemar_por_categoria(dados):
    resultado = {}
    categorias = dados['categoria_real'].unique()
    
    for categoria in categorias:
        acumulador = {}
        dados_categoria = dados[dados['categoria_real'] == categoria]
        acerto_a = (dados_categoria["categoria_real"] == dados_categoria["pred_modelo_a"]).astype(int)
        acerto_b = (dados_categoria["categoria_real"] == dados_categoria["pred_modelo_b"]).astype(int)

        b = ((acerto_a == 1) & (acerto_b == 0)).sum()
        c = ((acerto_a == 0) & (acerto_b == 1)).sum() 

        ambos_certos = ((acerto_a == 1) & (acerto_b == 1)).sum()
        ambos_errados = ((acerto_a == 0) & (acerto_b == 0)).sum()

        tabela = [[ambos_certos, b],
                  [c, ambos_errados]]
        acumulador["b"] = b
        acumulador["c"] = c
        resultado_categoria = mcnemar(tabela, exact=False)
        acumulador['estatistica'] = resultado_categoria.statistic
        acumulador['p-valor'] = resultado_categoria.pvalue
        resultado[categoria] = acumulador
    return resultado

In [60]:
mcnemar_por_categoria(dados)

{'barulho_perturbacao': {'b': np.int64(32),
  'c': np.int64(74),
  'estatistica': np.float64(15.858490566037736),
  'p-valor': np.float64(6.825958578031935e-05)},
 'iluminacao_publica': {'b': np.int64(93),
  'c': np.int64(213),
  'estatistica': np.float64(46.27777777777778),
  'p-valor': np.float64(1.0262128869361068e-11)},
 'coleta_lixo': {'b': np.int64(53),
  'c': np.int64(142),
  'estatistica': np.float64(39.712820512820514),
  'p-valor': np.float64(2.9418797644881963e-10)},
 'estacionamento_irregular': {'b': np.int64(39),
  'c': np.int64(70),
  'estatistica': np.float64(8.256880733944953),
  'p-valor': np.float64(0.004059782526314916)},
 'esgoto_vazamento': {'b': np.int64(47),
  'c': np.int64(240),
  'estatistica': np.float64(128.44599303135888),
  'p-valor': np.float64(8.965387205865398e-30)},
 'buraco_via': {'b': np.int64(64),
  'c': np.int64(155),
  'estatistica': np.float64(36.986301369863014),
  'p-valor': np.float64(1.189620965547304e-09)},
 'poda_arvore': {'b': np.int64(167)

| Categoria | Vencedor | b (A>B) | c (B>A) | Margem | Estatística | p-valor |
|---|:---:|---:|---:|---:|---:|---:|
| esgoto_vazamento | B | 47 | 240 | 193 | 128.45 | 8.97e-30 |
| iluminacao_publica | B | 93 | 213 | 120 | 46.28 | 1.03e-11 |
| poda_arvore | A | 167 | 53 | 114 | 58.04 | 2.57e-14 |
| buraco_via | B | 64 | 155 | 91 | 36.99 | 1.19e-09 |
| coleta_lixo | B | 53 | 142 | 89 | 39.71 | 2.94e-10 |
| barulho_perturbacao | B | 32 | 74 | 42 | 15.86 | 6.83e-05 |
| estacionamento_irregular | B | 39 | 70 | 31 | 8.26 | 0.00406 |
| sinalizacao | B | 22 | 44 | 22 | 6.68 | 0.00974 |

Analisando os resultados da tabela, é possível perceber que o modelo B mantém a tendência de performar melhor nas categorias individuais, com exceção da poda de árvores, na qual o modelo A se saiu melhor.  

Vale Reessaltar também que o p-value em todos os casos mostrou-se muito muito baixo, o maior deles mal chegava a 1% (sinalizacao), portanto mesmo utilizando um alpha conservador (1%), continuaríamos negando a hipótese nula.

Reomendação: Dado o comparativo de desempenho, é possível perceber que o modelo B performa melhor em todos os cenários exceto o da poda de árvores, portanto é recomendada a troca do modelo, uma vez que esse categoria reponde a menos de 10% dos chamados. Para mitigar eventuais problemas oriundos da deficiência do modelo B nessa categoria, o que pode ser feito é manter os dois modelos (A e B) online e levar em consideração a classificação do modelo A quando a saída dele for poda_arvore. Vale ressaltar que 80% das vezes que o modelo A classificou um chamado como poda de árvores ele estava certo, por isso é bastante razoável tratar o output dele como especialista nessas situações.